##**네이버 데이터 수집 파이프 라인**

1.   영화 공식 영화를 두고 실제 사람들이 어떤식으로 검색하는지 탐색 후 수동 수집(예 : 귀멸의 칼날: 상현집결, 그리고 도공 마을로 -> 귀칼 도공 마을, 귀멸의 칼날 도공의 마을)


*  **대상** : 세미콜론(:)이 있는 영화, 시리즈 영화, 제목이 긴 영화, 애니메이션 영화


2.   영화 기준 D-7~D+7구간 영화명 단독(별칭 포함) 검색량 수집



*   결과값 해석

      평균검색지수 : 영화가 15일 내내 얼마나 꾸준히 검색되었는지를 보는 비율

     (평균이 높을 수록 검색이 정점일에만 몰리지 않고 고르게 분포)
  
    - 영화 A: [3, 5, 8, 12, 100, 90, 50, ...] → 평균 35
    - 영화 B: [40, 50, 60, 80, 100, 85, 70, ...] → 평균 65
    
      --> A는 개봉일에만 반짝 검색되고 잊혀진 영화, B는 개봉 전부터 후까지 꾸준히 화제인 영화



##**네이버 검색량 130편 수집(영화명 단독 별칭 그룹도 포함)**

In [3]:
import requests
import json
import pandas as pd
import time
from datetime import datetime, timedelta
from google.colab import files

In [4]:
# ============================================================
# 셀 2. API 키 입력 + 키워드 사전 업로드
# ============================================================

# 본인 API 키로 교체
CLIENT_ID = "6PdDBbMD_m95LowDRESD"
CLIENT_SECRET = "Y_az4bbLpQ"

# 영화_검색키워드_사전.xlsx 파일 업로드
print(" 영화_검색키워드_사전.xlsx 파일을 업로드해주세요...")
uploaded = files.upload()

# 업로드된 파일명 자동 감지
file_name = list(uploaded.keys())[0]
print(f"\n 업로드 완료: {file_name}")

# 엑셀 읽기 (헤더가 2번째 행에 있으니 header=1)
movies_df = pd.read_excel(file_name, header=1)

# 개봉일을 datetime으로 변환
movies_df['개봉일'] = pd.to_datetime(movies_df['개봉일'].astype(str), format='%Y%m%d')

print(f"\n 영화 데이터 미리보기:")
print(f"   총 영화 수: {len(movies_df)}편")
print(f"   체급 분포: {movies_df['체급'].value_counts().to_dict()}")
print(f"   별칭 있는 영화: {(movies_df['키워드_개수']>=2).sum()}편")
print(f"\n 상위 3행:")
movies_df.head(3)

 영화_검색키워드_사전.xlsx 파일을 업로드해주세요...


Saving 영화_검색키워드_사전 (2).xlsx to 영화_검색키워드_사전 (2).xlsx

 업로드 완료: 영화_검색키워드_사전 (2).xlsx

 영화 데이터 미리보기:
   총 영화 수: 130편
   체급 분포: {'중형': 102, '대형': 14, '중대형': 14}
   별칭 있는 영화: 47편

 상위 3행:


,영화명_원본,체급,개봉일,검색키워드_1,검색키워드_2,검색키워드_3,검색키워드_4,검색키워드_5,키워드_개수
0,A MINECRAFT MOVIE 마인크래프트 무비,중형,2025-04-26,A MINECRAFT MOVIE 마인크래프트 무비,마인크래프트 무비,마인크래프트 영화,마크 영화,NaN,4
1,"귀멸의 칼날: 상현집결, 그리고 도공 마을로",중형,2023-03-02,"귀멸의 칼날 상현집결, 그리고 도공 마을로",귀멸의 칼날 상현6,귀멸의 칼날 도공마을,귀멸의 칼날 도공마을편,NaN,4
2,명탐정 코난: 척안의 잔상,중형,2025-07-16,명탐정 코난 척안의 잔상,코난 척안의 잔상,코난 28기,명탐정 코난 28기,NaN,4


In [5]:
# ============================================================
# 셀 3. 네이버 검색량 수집 함수 정의
# ============================================================

def get_naver_trend(keywords, group_name, start_date, end_date):
    """
    네이버 데이터랩에서 키워드 그룹의 일별 검색 트렌드를 가져온다.

    파라미터:
        keywords (list): 같은 영화를 가리키는 키워드들 (예: ["파묘", "영화 파묘"])
        group_name (str): 그룹 이름 (영화 원본명)
        start_date (str): 시작일 "YYYY-MM-DD"
        end_date (str): 종료일 "YYYY-MM-DD"

    반환:
        DataFrame 또는 None
    """
    url = "https://openapi.naver.com/v1/datalab/search"
    headers = {
        "X-Naver-Client-Id": CLIENT_ID,
        "X-Naver-Client-Secret": CLIENT_SECRET,
        "Content-Type": "application/json"
    }

    # 네이버 API는 그룹당 최대 5개 키워드까지 허용
    body = {
        "startDate": start_date,
        "endDate": end_date,
        "timeUnit": "date",
        "keywordGroups": [
            {
                "groupName": group_name,
                "keywords": keywords[:5]  # 최대 5개까지
            }
        ]
    }

    try:
        response = requests.post(url, headers=headers, data=json.dumps(body), timeout=10)

        if response.status_code != 200:
            print(f"    API 에러 ({response.status_code}): {response.text[:100]}")
            return None

        results = response.json()['results'][0]['data']
        if len(results) == 0:
            print(f"    검색 결과 없음")
            return None

        df = pd.DataFrame(results)
        df.columns = ['날짜', '검색지수']
        df['날짜'] = pd.to_datetime(df['날짜'])
        return df

    except Exception as e:
        print(f"    예외 발생: {e}")
        return None


def collect_movie(row):
    """
    영화 1편의 개봉 ±7일 검색량을 수집한다.

    파라미터:
        row (pd.Series): 엑셀의 한 행

    반환:
        DataFrame 또는 None
    """
    movie_name = row['영화명_원본']
    release_date = row['개봉일']

    # 키워드 리스트 만들기 (빈 칸 제외)
    keywords = []
    for i in range(1, 6):
        kw = row[f'검색키워드_{i}']
        if pd.notna(kw) and str(kw).strip():
            keywords.append(str(kw).strip())

    # 날짜 범위 계산
    start_date = (release_date - timedelta(days=7)).strftime('%Y-%m-%d')
    end_date = (release_date + timedelta(days=7)).strftime('%Y-%m-%d')

    # API 호출
    df = get_naver_trend(keywords, movie_name, start_date, end_date)

    if df is not None:
        df['영화명'] = movie_name
        df['체급'] = row['체급']
        df['개봉일'] = release_date
        df['사용된_키워드_개수'] = len(keywords)
        # 시점 구분
        df['시점'] = df['날짜'].apply(
            lambda x: '개봉전' if x < release_date
                     else ('개봉일' if x == release_date else '개봉후')
        )

    return df


print(" 함수 정의 완료")

 함수 정의 완료


In [6]:
# ============================================================
# 셀 4. 먼저 5편으로 테스트 ⭐
# ============================================================

print(f"{'='*60}")
print(f" 테스트: 별칭 있는 영화 3편 + 단독 검색 2편")
print(f"{'='*60}\n")

# 별칭 있는 영화 3편 + 단독 영화 2편 섞어서 테스트
test_aliased = movies_df[movies_df['키워드_개수']>=2].head(3)
test_single = movies_df[movies_df['키워드_개수']==1].head(2)
test_movies = pd.concat([test_aliased, test_single]).reset_index(drop=True)

test_results = []

for idx, row in test_movies.iterrows():
    keywords = [row[f'검색키워드_{i}'] for i in range(1,6)
                if pd.notna(row[f'검색키워드_{i}']) and str(row[f'검색키워드_{i}']).strip()]

    print(f" [{idx+1}/{len(test_movies)}] {row['영화명_원본']} ({row['체급']})")
    print(f"   키워드: {keywords}")
    print(f"   개봉일: {row['개봉일'].strftime('%Y-%m-%d')}")

    result = collect_movie(row)

    if result is not None:
        avg_idx = result['검색지수'].mean()
        max_idx = result['검색지수'].max()
        print(f"    수집 성공 — 평균 검색지수: {avg_idx:.1f}, 최대: {max_idx:.1f}")
        test_results.append(result)
    else:
        print(f"    수집 실패")

    time.sleep(0.5)  # API 부하 방지
    print()

# 결과 합치기
if test_results:
    test_df = pd.concat(test_results, ignore_index=True)
    print(f"{'='*60}")
    print(f" 테스트 완료! 총 {len(test_df)}행 수집")
    print(f"{'='*60}\n")
    print("--- 영화별 요약 ---")
    summary = test_df.groupby(['영화명','체급']).agg(
        평균검색지수=('검색지수', 'mean'),
        최대검색지수=('검색지수', 'max'),
        수집일수=('날짜', 'count')
    ).round(2)
    print(summary)

 테스트: 별칭 있는 영화 3편 + 단독 검색 2편

 [1/5] A MINECRAFT MOVIE 마인크래프트 무비 (중형)
   키워드: ['A MINECRAFT MOVIE 마인크래프트 무비', '마인크래프트 무비', '마인크래프트 영화', '마크 영화']
   개봉일: 2025-04-26
    수집 성공 — 평균 검색지수: 47.4, 최대: 100.0

 [2/5] 귀멸의 칼날: 상현집결, 그리고 도공 마을로 (중형)
   키워드: ['귀멸의 칼날 상현집결, 그리고 도공 마을로', '귀멸의 칼날 상현6', '귀멸의 칼날 도공마을', '귀멸의 칼날 도공마을편']
   개봉일: 2023-03-02
    수집 성공 — 평균 검색지수: 45.7, 최대: 100.0

 [3/5] 명탐정 코난: 척안의 잔상 (중형)
   키워드: ['명탐정 코난 척안의 잔상', '코난 척안의 잔상', '코난 28기', '명탐정 코난 28기']
   개봉일: 2025-07-16
    수집 성공 — 평균 검색지수: 48.0, 최대: 100.0

 [4/5] 밀수 (대형)
   키워드: ['밀수']
   개봉일: 2023-07-26
    수집 성공 — 평균 검색지수: 50.5, 최대: 100.0

 [5/5] 범죄도시3 (대형)
   키워드: ['범죄도시3']
   개봉일: 2023-05-31
    수집 성공 — 평균 검색지수: 66.9, 최대: 100.0

 테스트 완료! 총 75행 수집

--- 영화별 요약 ---
                                평균검색지수  최대검색지수  수집일수
영화명                         체급                      
A MINECRAFT MOVIE 마인크래프트 무비 중형   47.41   100.0    15
귀멸의 칼날: 상현집결, 그리고 도공 마을로    중형   45.72   100.0    15
명탐정 코난: 척안의 잔상              중형   48.02   100.0    

##**결과 값 해석**
- 평균검색지수 : 영화가 15일 내내 얼마나 꾸준히 검색되었는지를 보는 비율

   (평균이 높을 수록 검색이 정점일에만 몰리지 않고 고르게 분포)
  
    - 영화 A: [3, 5, 8, 12, 100, 90, 50, ...] → 평균 35
    - 영화 B: [40, 50, 60, 80, 100, 85, 70, ...] → 평균 65
    
      --> A는 개봉일에만 반짝 검색되고 잊혀진 영화, B는 개봉 전부터 후까지 꾸준히 화제인 영화

In [8]:
# ============================================================
# 셀 5. 130편 전체 자동 수집 (약 15분)
# ============================================================

print(f"{'='*60}")
print(f" 130편 전체 수집 시작 (영화명 + 별칭 그룹)")
print(f"   예상 소요 시간: 약 15분")
print(f"{'='*60}\n")

all_results = []
failed_movies = []
start_time = time.time()

for idx, row in movies_df.iterrows():
    if idx % 10 == 0:
        elapsed = time.time() - start_time
        print(f" [{idx}/{len(movies_df)}] 진행 중... (경과: {elapsed/60:.1f}분)")

    result = collect_movie(row)

    if result is not None:
        all_results.append(result)
    else:
        failed_movies.append(row['영화명_원본'])
        print(f"    실패: {row['영화명_원본']}")

    time.sleep(0.5)

# 결과 합치기
final_df = pd.concat(all_results, ignore_index=True)
final_df = final_df[['영화명', '체급', '개봉일', '날짜', '시점',
                     '검색지수', '사용된_키워드_개수']]

elapsed_total = time.time() - start_time
print(f"\n{'='*60}")
print(f" 수집 완료!")
print(f"{'='*60}")
print(f"    성공: {len(movies_df) - len(failed_movies)}편")
print(f"    실패: {len(failed_movies)}편")
print(f"    총 수집 데이터: {len(final_df)}행")
print(f"     총 소요 시간: {elapsed_total/60:.1f}분")

if failed_movies:
    print(f"\n실패한 영화:")
    for m in failed_movies:
        print(f"   - {m}")

# 저장 + 다운로드
output_filename = 'naver_검색량_단독수집.xlsx'  # 파일명에 "단독수집" 명시
final_df.to_excel(output_filename, index=False)
print(f"\n 저장 완료: {output_filename}")
print(f" 자동 다운로드 시작...")
files.download(output_filename)

 130편 전체 수집 시작 (영화명 + 별칭 그룹)
   예상 소요 시간: 약 15분

 [0/130] 진행 중... (경과: 0.0분)
 [10/130] 진행 중... (경과: 0.2분)
 [20/130] 진행 중... (경과: 0.3분)
 [30/130] 진행 중... (경과: 0.5분)
 [40/130] 진행 중... (경과: 0.6분)
 [50/130] 진행 중... (경과: 0.8분)
 [60/130] 진행 중... (경과: 1.0분)
 [70/130] 진행 중... (경과: 1.1분)
 [80/130] 진행 중... (경과: 1.3분)
 [90/130] 진행 중... (경과: 1.4분)
 [100/130] 진행 중... (경과: 1.6분)
 [110/130] 진행 중... (경과: 1.8분)
 [120/130] 진행 중... (경과: 1.9분)

 수집 완료!
    성공: 130편
    실패: 0편
    총 수집 데이터: 1950행
     총 소요 시간: 2.1분

 저장 완료: naver_검색량_단독수집.xlsx
 자동 다운로드 시작...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

##**네이버 검색 데이터 수집 - 검증형 키워드**

 - 검증형 검색 : 이미 영화를 본 사람의 후기를, 아직 안 본 사람이 찾아보는 행동.

    검증 신호가 강하면 2주차 생존율이 높다라는 메시지와 맞물리는 것을 반영함.



 - 검증형 키워드 분류(4개 카테고리)

   - "본 사람의 평가를 찾는다"는 의도의 키워드로 통일함.
   - 가장 대표적인 검증 행동.
   - API 한도 안전 : 키워드 그룹 4개는 네이버 한도(5개) 안에 안전하게 들어감.

**가장 직접적인 검증 키워드**

```
영화명 + 후기
```

   **비평,평가 정보 탐색**

```
영화명 + 리뷰
```

   **점수 기반 평가 확인**

```
영화명 + 평점
```

   **직접 본 사람의 짧은 평가**

```
영화명 + 관람평
```



*   **검색 구간 (개봉 당일~개봉 후 7일)**
*   **영화 제목이 한글자 두글자인 경우 "영화명 + 영화"별도 키워드를 붙여 수집진행**



In [9]:
# ============================================================
# 셀 1. 필요한 라이브러리 임포트
# ============================================================

import requests
import json
import pandas as pd
import time
from datetime import datetime, timedelta
from google.colab import files

print(" 라이브러리 임포트 완료")

 라이브러리 임포트 완료


셀 2. API 키 + 본인 대표키워드 파일 업로드

In [10]:
# ============================================================
# 셀 2. API 키 입력 + 영화_대표키워드.xlsx 업로드
# ============================================================

CLIENT_ID = "6PdDBbMD_m95LowDRESD"
CLIENT_SECRET = "Y_az4bbLpQ"

print(" 영화_대표키워드.xlsx 파일을 업로드해주세요...")
uploaded = files.upload()
file_name = list(uploaded.keys())[0]
print(f"\n 업로드 완료: {file_name}")

#  파일은 헤더가 2번째 줄(header=1)에 있음
movies_df = pd.read_excel(file_name, header=1)
movies_df.columns = ['영화명_원본', '체급', '개봉일', '검색키워드_1']
movies_df['개봉일'] = pd.to_datetime(movies_df['개봉일'].astype(str), format='%Y%m%d')

# 검색키워드_1의 양 끝 공백 제거 (혹시 모를 공백 처리)
movies_df['검색키워드_1'] = movies_df['검색키워드_1'].str.strip()

print(f"\n 영화 데이터:")
print(f"   총 영화 수: {len(movies_df)}편")
print(f"   체급 분포: {movies_df['체급'].value_counts().to_dict()}")
print(f"   대표키워드 평균 길이: {movies_df['검색키워드_1'].str.len().mean():.1f}자")

print(f"\n 상위 5행:")
movies_df.head()

 영화_대표키워드.xlsx 파일을 업로드해주세요...


Saving 영화_대표키워드.xlsx to 영화_대표키워드.xlsx

 업로드 완료: 영화_대표키워드.xlsx

 영화 데이터:
   총 영화 수: 130편
   체급 분포: {'중형': 102, '대형': 14, '중대형': 14}
   대표키워드 평균 길이: 6.7자

 상위 5행:


,영화명_원본,체급,개봉일,검색키워드_1
0,A MINECRAFT MOVIE 마인크래프트 무비,중형,2025-04-26,마인크래프트 영화
1,"귀멸의 칼날: 상현집결, 그리고 도공 마을로",중형,2023-03-02,귀멸의 칼날 상현집결
2,명탐정 코난: 척안의 잔상,중형,2025-07-16,코난 척안의 잔상
3,극장판 귀멸의 칼날: 무한성편,대형,2025-08-22,귀멸의 칼날 무한성
4,가디언즈 오브 갤럭시: Volume 3,중대형,2023-05-03,가디언즈 오브 갤럭시3


셀 3. 검증형 수집 함수

In [11]:
# ============================================================
# 셀 3. 검증형 키워드 수집 함수
# ============================================================

#  검증형 키워드 4종 — 발표 핵심 정의
VERIFICATION_SUFFIXES = ['후기', '리뷰', '평점', '관람평']


def get_naver_trend(keywords, group_name, start_date, end_date):
    """네이버 데이터랩에서 키워드 그룹의 일별 검색 트렌드를 가져온다."""
    url = "https://openapi.naver.com/v1/datalab/search"
    headers = {
        "X-Naver-Client-Id": CLIENT_ID,
        "X-Naver-Client-Secret": CLIENT_SECRET,
        "Content-Type": "application/json"
    }
    body = {
        "startDate": start_date,
        "endDate": end_date,
        "timeUnit": "date",
        "keywordGroups": [
            {
                "groupName": group_name,
                "keywords": keywords[:5]  # 네이버 한도 5개
            }
        ]
    }

    try:
        response = requests.post(url, headers=headers, data=json.dumps(body), timeout=10)
        if response.status_code != 200:
            print(f"    API 에러 ({response.status_code}): {response.text[:100]}")
            return None

        results = response.json()['results'][0]['data']
        if len(results) == 0:
            return None  # 검증 검색량 없음 (정상)

        df = pd.DataFrame(results)
        df.columns = ['날짜', '검증_검색지수']
        df['날짜'] = pd.to_datetime(df['날짜'])
        return df

    except Exception as e:
        print(f"    예외 발생: {e}")
        return None


def collect_verification(row):
    """
    영화 1편의 검증형 키워드 검색량 수집.
    검증 그룹 = [대표키워드 + 후기, 리뷰, 평점, 관람평]
    """
    movie_name = row['영화명_원본']
    release_date = row['개봉일']
    representative = row['검색키워드_1']

    if pd.isna(representative) or not str(representative).strip():
        return None

    representative = str(representative).strip()

    # 검증형 키워드 그룹 생성
    verification_keywords = [f"{representative} {suffix}" for suffix in VERIFICATION_SUFFIXES]

    #  변경 부분: 개봉일부터 시작
    start_date = release_date.strftime('%Y-%m-%d')           # 기존: -7일
    end_date = (release_date + timedelta(days=7)).strftime('%Y-%m-%d')

    df = get_naver_trend(verification_keywords, movie_name, start_date, end_date)


    if df is not None:
        df['영화명'] = movie_name
        df['체급'] = row['체급']
        df['개봉일'] = release_date
        df['대표키워드'] = representative
        df['검증_키워드_그룹'] = ', '.join(verification_keywords)
        df['시점'] = df['날짜'].apply(
            lambda x: '개봉일' if x == release_date else '개봉후'
        )

    return df


print(" 함수 정의 완료")
print(f"   검증형 키워드 4종: {VERIFICATION_SUFFIXES}")

# 예시 출력
example = movies_df.iloc[0]
example_kw = [f"{example['검색키워드_1']} {s}" for s in VERIFICATION_SUFFIXES]
print(f"\n   예시: '{example['영화명_원본']}'의 검증 그룹")
print(f"        → {example_kw}")

 함수 정의 완료
   검증형 키워드 4종: ['후기', '리뷰', '평점', '관람평']

   예시: 'A MINECRAFT MOVIE 마인크래프트 무비'의 검증 그룹
        → ['마인크래프트 영화 후기', '마인크래프트 영화 리뷰', '마인크래프트 영화 평점', '마인크래프트 영화 관람평']


셀 4. 5편 테스트

In [12]:
# ============================================================
# 셀 4. 5편 테스트
# ============================================================

print(f"{'='*60}")
print(f" 검증형 키워드 5편 테스트")
print(f"{'='*60}\n")

# 다양한 케이스 섞어서 테스트 (긴 키워드 + 짧은 키워드)
test_indices = [0, 1, 2, 60, 65]  # 본인 파일에서 다양한 위치
test_movies = movies_df.iloc[test_indices].reset_index(drop=True)

test_results = []

for idx, row in test_movies.iterrows():
    representative = row['검색키워드_1']
    verification_kws = [f"{representative} {suffix}" for suffix in VERIFICATION_SUFFIXES]

    print(f" [{idx+1}/5] {row['영화명_원본']} ({row['체급']})")
    print(f"   대표키워드: '{representative}'")
    print(f"   검증 그룹: {verification_kws}")
    print(f"   개봉일: {row['개봉일'].strftime('%Y-%m-%d')}")

    result = collect_verification(row)

    if result is not None:
        avg_idx = result['검증_검색지수'].mean()
        max_idx = result['검증_검색지수'].max()
        zero_count = (result['검증_검색지수']==0).sum()
        print(f"    수집 성공 — 평균: {avg_idx:.1f}, 최대: {max_idx:.1f}, 0인 날: {zero_count}일")
        test_results.append(result)
    else:
        print(f"    검증 검색량이 너무 적어 데이터 없음")

    time.sleep(0.5)
    print()

if test_results:
    test_df_v = pd.concat(test_results, ignore_index=True)
    print(f"{'='*60}")
    print(f" 테스트 완료! 총 {len(test_df_v)}행 수집")
    print(f"{'='*60}\n")
    print("--- 영화별 시점별 검증_검색지수 평균 ---")
    summary = test_df_v.groupby(['영화명','체급','시점'])['검증_검색지수'].mean().round(1).unstack()
    print(summary)

 검증형 키워드 5편 테스트

 [1/5] A MINECRAFT MOVIE 마인크래프트 무비 (중형)
   대표키워드: '마인크래프트 영화'
   검증 그룹: ['마인크래프트 영화 후기', '마인크래프트 영화 리뷰', '마인크래프트 영화 평점', '마인크래프트 영화 관람평']
   개봉일: 2025-04-26
    수집 성공 — 평균: 27.9, 최대: 100.0, 0인 날: 0일

 [2/5] 귀멸의 칼날: 상현집결, 그리고 도공 마을로 (중형)
   대표키워드: '귀멸의 칼날 상현집결'
   검증 그룹: ['귀멸의 칼날 상현집결 후기', '귀멸의 칼날 상현집결 리뷰', '귀멸의 칼날 상현집결 평점', '귀멸의 칼날 상현집결 관람평']
   개봉일: 2023-03-02
    수집 성공 — 평균: 41.7, 최대: 100.0, 0인 날: 0일

 [3/5] 명탐정 코난: 척안의 잔상 (중형)
   대표키워드: '코난 척안의 잔상'
   검증 그룹: ['코난 척안의 잔상 후기', '코난 척안의 잔상 리뷰', '코난 척안의 잔상 평점', '코난 척안의 잔상 관람평']
   개봉일: 2025-07-16
    수집 성공 — 평균: 72.1, 최대: 100.0, 0인 날: 0일

 [4/5] 야당 (중대형)
   대표키워드: '야당'
   검증 그룹: ['야당 후기', '야당 리뷰', '야당 평점', '야당 관람평']
   개봉일: 2025-04-16
    수집 성공 — 평균: 74.9, 최대: 100.0, 0인 날: 0일

 [5/5] 하얼빈 (중대형)
   대표키워드: '하얼빈'
   검증 그룹: ['하얼빈 후기', '하얼빈 리뷰', '하얼빈 평점', '하얼빈 관람평']
   개봉일: 2024-12-24
    수집 성공 — 평균: 52.6, 최대: 100.0, 0인 날: 0일

 테스트 완료! 총 40행 수집

--- 영화별 시점별 검증_검색지수 평균 ---
시점                                 개봉일   개봉후
영화명       

셀 5. 130편 전체 수집

In [14]:
# ============================================================
# 셀 5. 130편 전체 검증형 키워드 수집 (약 15분)
# ============================================================

print(f"{'='*60}")
print(f" 130편 검증형 키워드 수집 시작")
print(f"   키워드 정의: {VERIFICATION_SUFFIXES}")
print(f"   예상 소요 시간: 약 15분")
print(f"{'='*60}\n")

all_results = []
no_data_movies = []
failed_movies = []
start_time = time.time()

for idx, row in movies_df.iterrows():
    if idx % 10 == 0:
        elapsed = time.time() - start_time
        print(f" [{idx}/{len(movies_df)}] 진행 중... (경과: {elapsed/60:.1f}분)")

    try:
        result = collect_verification(row)
        if result is not None:
            all_results.append(result)
        else:
            no_data_movies.append(row['영화명_원본'])
    except Exception as e:
        failed_movies.append((row['영화명_원본'], str(e)))
        print(f"    에러: {row['영화명_원본']} ({e})")

    time.sleep(0.5)

# 결과 합치기
if all_results:
    final_df = pd.concat(all_results, ignore_index=True)
    final_df = final_df[['영화명', '체급', '개봉일', '날짜', '시점',
                         '검증_검색지수', '대표키워드', '검증_키워드_그룹']]
else:
    final_df = pd.DataFrame()

elapsed_total = time.time() - start_time
print(f"\n{'='*60}")
print(f" 수집 완료!")
print(f"{'='*60}")
print(f"    검증 데이터 수집됨: {len(movies_df) - len(no_data_movies) - len(failed_movies)}편")
print(f"    검증 데이터 없음: {len(no_data_movies)}편")
print(f"    에러 발생: {len(failed_movies)}편")
print(f"    총 수집 행 수: {len(final_df)}행")
print(f"    소요 시간: {elapsed_total/60:.1f}분")

# 검증 데이터 없는 영화 분석 (체급별)
if no_data_movies:
    no_data_df = movies_df[movies_df['영화명_원본'].isin(no_data_movies)]
    print(f"\n--- 검증 데이터 없는 영화의 체급 분포 ---")
    print(no_data_df['체급'].value_counts().to_dict())
    print("\n 발표 메시지로 활용 가능: '입소문이 시작되기 전에 사라진 영화들'")

if failed_movies:
    print(f"\n에러 영화:")
    for m, e in failed_movies:
        print(f"   - {m}: {e}")

# 엑셀 저장 (2시트: 데이터 있는 영화 + 데이터 없는 영화)
output_filename = 'naver_검색량_검증형수집.xlsx'

with pd.ExcelWriter(output_filename) as writer:
    final_df.to_excel(writer, sheet_name='검증_검색량', index=False)

    if no_data_movies:
        no_data_export = movies_df[movies_df['영화명_원본'].isin(no_data_movies)].copy()
        no_data_export['비고'] = '검증 검색량 부족 (입소문 약함 시그널)'
        no_data_export.to_excel(writer, sheet_name='검증_데이터_없음', index=False)

print(f"\n 저장 완료: {output_filename}")
print(f" 자동 다운로드 시작...")
files.download(output_filename)

 130편 검증형 키워드 수집 시작
   키워드 정의: ['후기', '리뷰', '평점', '관람평']
   예상 소요 시간: 약 15분

 [0/130] 진행 중... (경과: 0.0분)
 [10/130] 진행 중... (경과: 0.2분)
 [20/130] 진행 중... (경과: 0.3분)
 [30/130] 진행 중... (경과: 0.5분)
 [40/130] 진행 중... (경과: 0.6분)
 [50/130] 진행 중... (경과: 0.8분)
 [60/130] 진행 중... (경과: 1.0분)
 [70/130] 진행 중... (경과: 1.1분)
 [80/130] 진행 중... (경과: 1.3분)
 [90/130] 진행 중... (경과: 1.5분)
 [100/130] 진행 중... (경과: 1.6분)
 [110/130] 진행 중... (경과: 1.8분)
 [120/130] 진행 중... (경과: 2.0분)

 수집 완료!
    검증 데이터 수집됨: 127편
    검증 데이터 없음: 3편
    에러 발생: 0편
    총 수집 행 수: 1002행
    소요 시간: 2.1분

--- 검증 데이터 없는 영화의 체급 분포 ---
{'중형': 3}

 발표 메시지로 활용 가능: '입소문이 시작되기 전에 사라진 영화들'

 저장 완료: naver_검색량_검증형수집.xlsx
 자동 다운로드 시작...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

##**네이버 검증형 검색 데이터 결과해석**

```
검증 데이터 수집 -> 127편
```

```
검증 데이터 없음 -> 3편(모두 짱구, 장화신은 고양이 같은 일부 일본 애니)
```

```
총 데이터 행 개수 -> 1002행(128편x 약8일)
```

```
검색지수 0인 행 -> 0건
```
---

1.  **검증 데이터 없는  3편 분석**

    - 짱구는 못말려: 동물소환 닌자 배꼽수비대

    - 신차원! 짱구는 못말려: 더 무비





    - 장화신은 고양이: 끝내주는 모험

--> 세 편 모두 **저연령층 대상 일본/외국 애니메이션임.**

   - 저연령 대상 애니메이션은 아이위주로 보는 영화라 성인 관객 중심의 후기,리뷰 검색 행태가 잘 발생하지 않음.

2. 데이터에서 보이는 흥미로운 발견

**발견1. 검증 신호가 약한 영화들(하위10편)**

```
1위 마인크래프트 영화 (27.9)

2위 트랜스포머 비스트의 서막 (28.4)

3위 앤트맨과 와스프 퀀텀매니아 (30.8)

4위 그대들은 어떻게 살 것인가 (33.5)

5위 분노의 질주 라이드 오어 다이 (35.2)
```

모두 외화,시리즈 속편임. 한국 관객들에게 '검증 검색을 굳이 할 만큼의 화제성' 없는 영화들.


**발견2.검증 신호가 강한 영화들 (상위10편)**

```
1위 짱구 우리들의 공룡일기 (100.0)
2위 만약에 우리 (88.5)
3위 히트맨2 (83.9)
4위 엘리멘탈 (84.4)
5위 무파사 (81.6)
6위 슈퍼배드 4 (81.4)
7위 노이즈 (80.4)
8위 소방관 (78.9)
9위 청설 (77.9)
10위 밀수 (76.9)
```

한국적 화제성이 있는 영화 + 가족 단위 흥행작이 상위에 몰림.


**발견3. 시점별 평균이 의미하는 것**

- 개봉일 평균 88.4% : 거의 정점 - 모든 영화가 개봉일에 검증 검색이 폭발.

- 개봉 후 평균 57,4% : 평균 35%감소 - 1주일 만에 거의 절반 가까이 식음.

##**네이버 검색 데이터 수집 - 기대형 키워드**

 - 기대형 검색 : 아직 안 본 사람이, 영화를 볼지 말지 결정하기 위해 정보를 탐색하는 행동.

 - 기대형 키워드 분류(4개 카테고리)

**영상으로 미리보기 -가장 직접적**

```
영화명 + 예고편
```

   **일찍 보고 싶은 적극적 관심**

```
영화명 + 시사회
```

   **언제 볼 수 있는지 확인**

```
영화명 + 개봉일
```

   **누가 나오는지 확인**

```
영화명 + 출연 or 배우
```



*   **검색 구간 (개봉 7일 전~개봉 일)**



In [15]:
# ============================================================
# 셀 3. 기대형 키워드 수집 함수 정의
# ============================================================

#  기대형 키워드 4종
EXPECTATION_SUFFIXES = ['예고편', '시사회', '개봉일', '출연']


def get_naver_trend(keywords, group_name, start_date, end_date):
    """네이버 데이터랩에서 키워드 그룹의 일별 검색 트렌드를 가져온다."""
    url = "https://openapi.naver.com/v1/datalab/search"
    headers = {
        "X-Naver-Client-Id": CLIENT_ID,
        "X-Naver-Client-Secret": CLIENT_SECRET,
        "Content-Type": "application/json"
    }
    body = {
        "startDate": start_date,
        "endDate": end_date,
        "timeUnit": "date",
        "keywordGroups": [
            {
                "groupName": group_name,
                "keywords": keywords[:5]
            }
        ]
    }

    try:
        #  json=body 방식 (인코딩 안전)
        response = requests.post(url, headers=headers, json=body, timeout=10)

        if response.status_code != 200:
            print(f"    API 에러 ({response.status_code}): {response.text[:100]}")
            return None

        results = response.json()['results'][0]['data']
        if len(results) == 0:
            return None

        df = pd.DataFrame(results)
        df.columns = ['날짜', '기대_검색지수']  # 컬럼명에 "기대" 명시
        df['날짜'] = pd.to_datetime(df['날짜'])
        return df

    except Exception as e:
        print(f"    예외 발생: {e}")
        return None


def collect_expectation(row):
    """
    영화 1편의 기대형 키워드 검색량 수집.
    기대 그룹 = [대표키워드 + 예고편, 시사회, 개봉일, 출연]
    수집 범위: 개봉 -7일 ~ 개봉일 (안 본 사람의 정보 탐색 기간)
    """
    movie_name = row['영화명_원본']
    release_date = row['개봉일']
    representative = row['검색키워드_1']

    if pd.isna(representative) or not str(representative).strip():
        return None

    representative = str(representative).strip()

    # 기대형 키워드 그룹 생성
    expectation_keywords = [f"{representative} {suffix}" for suffix in EXPECTATION_SUFFIXES]

    #  날짜 범위: 개봉 -7일 ~ 개봉일 (검증형과 정반대)
    start_date = (release_date - timedelta(days=7)).strftime('%Y-%m-%d')
    end_date = release_date.strftime('%Y-%m-%d')

    # API 호출
    df = get_naver_trend(expectation_keywords, movie_name, start_date, end_date)

    if df is not None:
        df['영화명'] = movie_name
        df['체급'] = row['체급']
        df['개봉일'] = release_date
        df['대표키워드'] = representative
        df['기대_키워드_그룹'] = ', '.join(expectation_keywords)
        #  시점 구분: 개봉전 / 개봉일
        df['시점'] = df['날짜'].apply(
            lambda x: '개봉일' if x == release_date else '개봉전'
        )

    return df


print(" 기대형 함수 정의 완료")
print(f"   기대형 키워드 4종: {EXPECTATION_SUFFIXES}")
print(f"   수집 범위: 개봉 -7일 ~ 개봉일 (안 본 사람의 정보 탐색 기간)")

# 예시
example = movies_df.iloc[0]
example_kw = [f"{example['검색키워드_1']} {s}" for s in EXPECTATION_SUFFIXES]
print(f"\n   예시: '{example['영화명_원본']}'의 기대 그룹")
print(f"        → {example_kw}")

 기대형 함수 정의 완료
   기대형 키워드 4종: ['예고편', '시사회', '개봉일', '출연']
   수집 범위: 개봉 -7일 ~ 개봉일 (안 본 사람의 정보 탐색 기간)

   예시: 'A MINECRAFT MOVIE 마인크래프트 무비'의 기대 그룹
        → ['마인크래프트 영화 예고편', '마인크래프트 영화 시사회', '마인크래프트 영화 개봉일', '마인크래프트 영화 출연']


In [16]:
# ============================================================
# 셀 4. 기대형 키워드 5편 테스트
# ============================================================

print(f"{'='*60}")
print(f" 기대형 키워드 5편 테스트")
print(f"{'='*60}\n")

# 검증형과 동일한 5편으로 테스트 (비교 가능하게)
test_indices = [0, 1, 2, 60, 65]
test_movies = movies_df.iloc[test_indices].reset_index(drop=True)

test_results = []

for idx, row in test_movies.iterrows():
    representative = row['검색키워드_1']
    expectation_kws = [f"{representative} {suffix}" for suffix in EXPECTATION_SUFFIXES]

    print(f" [{idx+1}/5] {row['영화명_원본']} ({row['체급']})")
    print(f"   대표키워드: '{representative}'")
    print(f"   기대 그룹: {expectation_kws}")
    print(f"   개봉일: {row['개봉일'].strftime('%Y-%m-%d')}")

    result = collect_expectation(row)

    if result is not None:
        avg_idx = result['기대_검색지수'].mean()
        max_idx = result['기대_검색지수'].max()
        zero_count = (result['기대_검색지수']==0).sum()
        print(f"    수집 성공 — 평균: {avg_idx:.1f}, 최대: {max_idx:.1f}, 0인 날: {zero_count}일")
        test_results.append(result)
    else:
        print(f"    기대 검색량 데이터 없음")

    time.sleep(0.5)
    print()

if test_results:
    test_df_e = pd.concat(test_results, ignore_index=True)
    print(f"{'='*60}")
    print(f" 테스트 완료! 총 {len(test_df_e)}행 수집")
    print(f"{'='*60}\n")
    print("--- 영화별 시점별 기대_검색지수 평균 ---")
    summary = test_df_e.groupby(['영화명','체급','시점'])['기대_검색지수'].mean().round(1).unstack()
    print(summary)
    print("\n 해석: 개봉전 평균이 높을수록 '사전 기대 강한 영화'")
    print("       개봉일에 정점 찍는 게 일반적 패턴")

 기대형 키워드 5편 테스트

 [1/5] A MINECRAFT MOVIE 마인크래프트 무비 (중형)
   대표키워드: '마인크래프트 영화'
   기대 그룹: ['마인크래프트 영화 예고편', '마인크래프트 영화 시사회', '마인크래프트 영화 개봉일', '마인크래프트 영화 출연']
   개봉일: 2025-04-26
    수집 성공 — 평균: 29.8, 최대: 100.0, 0인 날: 0일

 [2/5] 귀멸의 칼날: 상현집결, 그리고 도공 마을로 (중형)
   대표키워드: '귀멸의 칼날 상현집결'
   기대 그룹: ['귀멸의 칼날 상현집결 예고편', '귀멸의 칼날 상현집결 시사회', '귀멸의 칼날 상현집결 개봉일', '귀멸의 칼날 상현집결 출연']
   개봉일: 2023-03-02
    수집 성공 — 평균: 48.7, 최대: 100.0, 0인 날: 0일

 [3/5] 명탐정 코난: 척안의 잔상 (중형)
   대표키워드: '코난 척안의 잔상'
   기대 그룹: ['코난 척안의 잔상 예고편', '코난 척안의 잔상 시사회', '코난 척안의 잔상 개봉일', '코난 척안의 잔상 출연']
   개봉일: 2025-07-16
    수집 성공 — 평균: 65.2, 최대: 100.0, 0인 날: 0일

 [4/5] 야당 (중대형)
   대표키워드: '야당'
   기대 그룹: ['야당 예고편', '야당 시사회', '야당 개봉일', '야당 출연']
   개봉일: 2025-04-16
    수집 성공 — 평균: 34.9, 최대: 100.0, 0인 날: 0일

 [5/5] 하얼빈 (중대형)
   대표키워드: '하얼빈'
   기대 그룹: ['하얼빈 예고편', '하얼빈 시사회', '하얼빈 개봉일', '하얼빈 출연']
   개봉일: 2024-12-24
    수집 성공 — 평균: 60.9, 최대: 100.0, 0인 날: 0일

 테스트 완료! 총 39행 수집

--- 영화별 시점별 기대_검색지수 평균 ---
시점                                 개봉일   개봉전


In [17]:
# ============================================================
# 셀 5. 130편 전체 기대형 키워드 수집 (약 12분)
# ============================================================

print(f"{'='*60}")
print(f" 130편 기대형 키워드 수집 시작")
print(f"   키워드: {EXPECTATION_SUFFIXES}")
print(f"   범위: 개봉 -7일 ~ 개봉일")
print(f"{'='*60}\n")

all_results = []
no_data_movies = []
failed_movies = []
start_time = time.time()

for idx, row in movies_df.iterrows():
    if idx % 10 == 0:
        elapsed = time.time() - start_time
        print(f" [{idx}/{len(movies_df)}] 진행 중... (경과: {elapsed/60:.1f}분)")

    try:
        result = collect_expectation(row)
        if result is not None:
            all_results.append(result)
        else:
            no_data_movies.append(row['영화명_원본'])
    except Exception as e:
        failed_movies.append((row['영화명_원본'], str(e)))
        print(f"    에러: {row['영화명_원본']} ({e})")

    time.sleep(0.5)

# 결과 합치기
if all_results:
    final_df = pd.concat(all_results, ignore_index=True)
    final_df = final_df[['영화명', '체급', '개봉일', '날짜', '시점',
                         '기대_검색지수', '대표키워드', '기대_키워드_그룹']]
else:
    final_df = pd.DataFrame()

elapsed_total = time.time() - start_time
print(f"\n{'='*60}")
print(f" 수집 완료!")
print(f"{'='*60}")
print(f"   기대 데이터 수집됨: {len(movies_df) - len(no_data_movies) - len(failed_movies)}편")
print(f"   기대 데이터 없음: {len(no_data_movies)}편")
print(f"   에러 발생: {len(failed_movies)}편")
print(f"   총 수집 행 수: {len(final_df)}행")
print(f"   소요 시간: {elapsed_total/60:.1f}분")

if no_data_movies:
    no_data_df = movies_df[movies_df['영화명_원본'].isin(no_data_movies)]
    print(f"\n--- 기대 데이터 없는 영화의 체급 분포 ---")
    print(no_data_df['체급'].value_counts().to_dict())

if failed_movies:
    print(f"\n에러 영화:")
    for m, e in failed_movies:
        print(f"   - {m}: {e}")

# 엑셀 저장
output_filename = 'naver_검색량_기대형수집.xlsx'

with pd.ExcelWriter(output_filename) as writer:
    final_df.to_excel(writer, sheet_name='기대_검색량', index=False)

    if no_data_movies:
        no_data_export = movies_df[movies_df['영화명_원본'].isin(no_data_movies)].copy()
        no_data_export['비고'] = '기대 검색량 부족 (사전 관심 약함 시그널)'
        no_data_export.to_excel(writer, sheet_name='기대_데이터_없음', index=False)

print(f"\n 저장 완료: {output_filename}")
print(f" 자동 다운로드 시작...")
files.download(output_filename)

 130편 기대형 키워드 수집 시작
   키워드: ['예고편', '시사회', '개봉일', '출연']
   범위: 개봉 -7일 ~ 개봉일

 [0/130] 진행 중... (경과: 0.0분)
 [10/130] 진행 중... (경과: 0.2분)
 [20/130] 진행 중... (경과: 0.3분)
 [30/130] 진행 중... (경과: 0.5분)
 [40/130] 진행 중... (경과: 0.6분)
 [50/130] 진행 중... (경과: 0.8분)
 [60/130] 진행 중... (경과: 1.0분)
 [70/130] 진행 중... (경과: 1.1분)
 [80/130] 진행 중... (경과: 1.3분)
 [90/130] 진행 중... (경과: 1.5분)
 [100/130] 진행 중... (경과: 1.6분)
 [110/130] 진행 중... (경과: 1.8분)
 [120/130] 진행 중... (경과: 1.9분)

 수집 완료!
   기대 데이터 수집됨: 125편
   기대 데이터 없음: 5편
   에러 발생: 0편
   총 수집 행 수: 979행
   소요 시간: 2.1분

--- 기대 데이터 없는 영화의 체급 분포 ---
{'중형': 4, '중대형': 1}

 저장 완료: naver_검색량_기대형수집.xlsx
 자동 다운로드 시작...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

##**네이버 검증형 검색 데이터 결과해석**

```
기대 데이터 수집 -> 125편
```

```
기대 데이터 없음 -> 5편
```

```
총 데이터 행 개수 -> 974행
```

```
검색지수 0인 행 -> 0건
```
---

1.  **기대 데이터 없는  3편 분석**

    - 짱구는 못말려: 동물소환 닌자 배꼽수비대

    - 신차원! 짱구는 못말려: 더 무비

    - 장화신은 고양이: 끝내주는 모험

--> 모두 저연령층 외화 애니메이션임.

### **추가 발견**
   
   - 더 퍼스트 슬램덩크 - 기대 데이터 없음

   - 극장판 짱구 우리들의 공룡일기 - 기대 데이터 없음.

슬램덩크는 검증 검색이 풍부 했지만 기대 검색이 없다는 점.

```
개봉 전 사람들이 '슬램덩크 영화 예고편'같은 검색 안 하고 그냥 봤다는 것.
```

```
만화 원작 충성팬이 많은 영화의 특이한 패턴.
```

2. 데이터에서 보이는 흥미로운 발견

**발견1. 검증/기대 비율의 의미**

영화|기대 평균|검증 평균|검증/기대 비율|
|------|---|---|---|
|야당|44.0|57.5|1.31|
|코난 척안의 잔상|65.2|72.1|1.11|
|마인크래프트|29.8|27.9|0.94|
|하얼빈|60.9|52.6|0.86|
|귀멸의 칼날|48.7|41.7|0.85|

비율>1.0
```
기대보다 검증이 강함 = " 실제로 보고 나서 후기 검색이 폭발한 영화" = 입소문이 강한 영화
```

비율<1.0
```
기대보다 검증이 약함 = "기대만큼 검증이 안따라온 영화" = 흥행 영화 악함.
```